# Vanilla RAG в Colab

Выберите GPU в настройках среды выполнения и нажмите «Выполнить всё». В первой ячейке дайте доступ к секрету `GROQ_API_KEY` и выберите `articles.json`, `questions.json` и `ground_truth.json` в одном диалоге. Файлы хранятся во временной среде Colab. В конце ноутбук скачает `rag-results.zip`.

In [ ]:
from pathlib import Path
from google.colab import files, userdata
from tempfile import TemporaryDirectory
import os
import subprocess
import sys

DATA = Path("/content/rag-data")
RESULTS = Path("/content/rag-results")
DATA_FILES = {"articles.json", "questions.json", "ground_truth.json"}
REPO = "https://github.com/semenovfilipp/rag.git"
BRANCH = "feature/vanilla-rag"
CHECKOUT = Path("/content/rag")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

if not CHECKOUT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO, str(CHECKOUT)], check=True)
else:
    subprocess.run(["git", "-C", str(CHECKOUT), "pull", "--ff-only", "origin", BRANCH], check=True)

def experiment(*args):
    subprocess.run([
        sys.executable, str(CHECKOUT / "scripts/experiment.py"),
        "--data", str(DATA), "--results", str(RESULTS), *args,
    ], check=True)

if not all((DATA / name).is_file() for name in DATA_FILES):
    print("Выберите три JSON-файла одновременно.")
    with TemporaryDirectory(dir="/content") as upload_dir:
        uploaded = files.upload(target_dir=upload_dir)
        uploaded_by_name = {Path(name).name: content for name, content in uploaded.items()}
        if uploaded_by_name.keys() != DATA_FILES:
            raise ValueError("Выберите articles.json, questions.json и ground_truth.json.")
        DATA.mkdir(exist_ok=True)
        for name, content in uploaded_by_name.items():
            (DATA / name).write_bytes(content)
    del uploaded, uploaded_by_name, content

experiment("validate")

In [ ]:
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "-r", str(CHECKOUT / "requirements.txt"), "vllm",
], check=True)

In [ ]:
import json
import time
from urllib.error import URLError
from urllib.request import urlopen

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LOG = Path("/content/vllm.log")

def vllm_ready():
    try:
        with urlopen("http://127.0.0.1:8000/v1/models", timeout=2) as response:
            return any(row["id"] == MODEL for row in json.load(response)["data"])
    except (URLError, TimeoutError):
        return False

if not vllm_ready():
    if "server" not in globals() or server.poll() is not None:
        with LOG.open("w") as log:
            server = subprocess.Popen([
                "vllm", "serve", MODEL, "--host", "127.0.0.1",
                "--max-model-len", "4096", "--dtype", "half",
                "--generation-config", "vllm",
            ], stdout=log, stderr=subprocess.STDOUT)
    deadline = time.monotonic() + 900
    while not vllm_ready():
        if server.poll() is not None or time.monotonic() >= deadline:
            raise RuntimeError(LOG.read_text(errors="replace")[-6000:])
        time.sleep(3)
print("vLLM ready")

In [ ]:
# Проверяем весь путь генерации на двух вопросах. Ответы сохраняются.
for mode in ("groq", "local", "rag"):
    experiment("run", mode, "--limit", "2")

In [ ]:
for mode in ("groq", "local", "rag"):
    experiment("run", mode)

experiment("evaluate")

In [ ]:
import json
from zipfile import ZIP_DEFLATED, ZipFile

archive = Path("/content/rag-results.zip")
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    bundle.write(RESULTS / "summary.json", "summary.json")
    for mode in ("groq", "local", "rag"):
        for filename in (f"{mode}.json", f"evaluation_{mode}.json"):
            payload = json.loads((RESULTS / filename).read_text(encoding="utf-8"))
            bundle.writestr(filename, json.dumps(payload["records"], ensure_ascii=False, indent=2))

files.download(str(archive))